In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.mslandcover.models import HighResolutionNet, get_cls_net
from src.mslandcover.config import HRNET_BASE_CONFIG

In [2]:
model = get_cls_net(config=HRNET_BASE_CONFIG)

In [9]:
test_batch = torch.rand(1, 3, 256, 256)
output = model(test_batch)
n_channels_total = 0
for i in output:
    n_channels_total += i.size(1)
print(n_channels_total)

720


In [ ]:
class DecoderLayer(nn.Module):
    
    def __init__(self, in_channels, out_channels, output_size=64):
        super(DecoderLayer, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.upsample = nn.Upsample(size=output_size, mode='bilinear', align_corners=True)
    
    def forward(self, x):
        
        x = self.upsample(x)
        x_res = F.relu(self.pointwise(x))
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x + x_res
        return x



class ConvClassifier(nn.Module):
    
    def __init__(self, in_channels, n_classes):
        
        super(ConvClassifier, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, n_classes, kernel_size=1)
    
    def forward(self, x):
        x = self.conv1(x)
        return x



class ImageDecoder(nn.Module):
    
    def __init__(self, in_channels, n_classes, scale_factor=4):
        super(ImageDecoder, self).__init__()
        
        self.n_layers = F.log2(torch.tensor(scale_factor)).int()
        self.layers = nn.ModuleList([])
        current_layer_channels = in_channels
        for _ in range(self.n_layers):
            self.layers.append(DecoderLayer(current_layer_channels, current_layer_channels // 2))
            current_layer_channels = current_layer_channels // 2
        
        self.classifier = ConvClassifier(current_layer_channels, n_classes)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x = self.classifier(x)
        return x



class ProjectionHead(nn.Module):
    
    def __init__(self, in_channels, hiddens, out_dim):
        super(ProjectionHead, self).__init__()
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        # self.hidden = 
    def forward(self, x):
        x = self.global_pool(x)
        for layer in self.hiddens:
            x = F.relu(layer(x))

In [1]:
class HRNetSegmentationModel(nn.Module):
    
    def __init__(self, config):
        super(HRNetSegmentationModel, self).__init__()
        self.config = config
        self.encoder = HighResolutionNet(config)
        self.decoder = ImageDecoder(720, config.n_classes)
    
    def forward(self, x):
        z_list = self.encoder(x)
        for z in z_list:
            print(z.size())
        z = torch.cat(z_list, dim=1)
        x = self.decoder(z)
        return x
    
    def load_encoder_weights(self, path):
        self.encoder.load_state_dict(torch.load(path))

NameError: name 'nn' is not defined